<a href="https://colab.research.google.com/github/sumankayal247/storing-shit/blob/main/colab_image_gen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Simple Text-to-Image Generation on Colab (SDXL) - Gradio UI  (JUST RUN ALL)

Runtime > Change runtime type > **T4 GPU** before running, then Runtime > Run all.

This notebook sets up a simple Gradio-based user interface for text-to-image generation using the Stable Diffusion XL model, running locally in your Colab environment. It provides a Colab-authenticated link or a public Gradio Share link to access the UI.

See the last cell for caveats (Colab free-tier limits, licenses, Drive caching).

## 0. (optional) cache models on Google Drive so you don't re-download every session

In [ ]:
# @title
from google.colab import drive
drive.mount('/content/drive')
import os
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'  # HuggingFace models cached here

Mounted at /content/drive


### Model Caching in Google Drive

By setting the `HF_HOME` environment variable to a path within your mounted Google Drive (`/content/drive/MyDrive/hf_cache`), the `diffusers` library will automatically handle caching:

*   **First Run**: If the Stable Diffusion XL model is not found in `/content/drive/MyDrive/hf_cache`, it will be downloaded from Hugging Face and stored in this location.
*   **Subsequent Runs**: In future sessions, if the model already exists in `/content/drive/MyDrive/hf_cache`, `diffusers` will load it directly from your Drive, saving download time and bandwidth.

This ensures that after the initial download, you won't have to re-download the model each time you run the notebook, as long as your Google Drive is successfully mounted.

## Previous ComfyUI Setup (no longer in use)

The following cells were part of the previous ComfyUI setup and are no longer required for the Gradio-based text-to-image generator. They have been commented out or converted to markdown for clarity.

In [ ]:
# @title
# Terminate ComfyUI process
try:
    if 'comfy' in locals() and comfy.poll() is None:
        comfy.terminate()
        comfy.wait(timeout=10) # Wait for termination
        print("ComfyUI process terminated.")
except NameError:
    print("ComfyUI process handle 'comfy' not found, it might already be stopped.")
except Exception as e:
    print(f"Error terminating ComfyUI process: {e}")

# Terminate Cloudflared process
try:
    if 'cloudflared_process' in locals() and cloudflared_process.poll() is None:
        cloudflared_process.terminate()
        cloudflared_process.wait(timeout=10)
        print("Cloudflared process terminated.")
except NameError:
    print("Cloudflared process handle 'cloudflared_process' not found, it might already be stopped.")
except Exception as e:
    print(f"Error terminating Cloudflared process: {e}")

import time
time.sleep(5) # Give processes a moment to fully shut down

### Set up a simple text-to-image generator using `diffusers` and `gradio`

This section will install the necessary libraries, load the Stable Diffusion XL model, and create a user-friendly Gradio interface for text-to-image generation.

In [ ]:
# @title
# 1. Install necessary libraries
!pip install -q diffusers transformers accelerate gradio

In [ ]:
# @title
import torch
from diffusers import DiffusionPipeline
import gradio as gr
import os

# Ensure HF_HOME is set for caching models if it was set in a previous cell
# If you uncommented and ran cell uZTJkSBJNBeY, models will be cached in Drive.
os.environ['HF_HOME'] = '/content/drive/MyDrive/hf_cache'


Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.
Flax classes are deprecated and will be removed in Diffusers v1.0.0. We recommend migrating to PyTorch classes or pinning your version of Diffusers.


In [ ]:
# @title
# 3. Load the Stable Diffusion XL pipeline
print("Loading Stable Diffusion XL model... This may take a few minutes.")

pipeline = DiffusionPipeline.from_pretrained("stabilityai/stable-diffusion-xl-base-1.0", torch_dtype=torch.float16)
pipeline.to("cuda")

print("Model loaded successfully!")

Loading Stable Diffusion XL model... This may take a few minutes.


Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/517 [00:00<?, ?it/s]

Model loaded successfully!


In [ ]:
# @title
import gradio as gr
import socket
import time

# 4. Define the inference function for Gradio
def generate_image(prompt, negative_prompt=None):
    if negative_prompt:
        image = pipeline(prompt, negative_prompt=negative_prompt).images[0]
    else:
        image = pipeline(prompt).images[0]
    return image

# 5. Create the Gradio interface
print("Setting up Gradio interface...")

iface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(lines=2, placeholder="Enter your prompt here...", label="Prompt"),
        gr.Textbox(
            lines=2,
            placeholder="Enter negative prompt here (e.g., blurry, low quality)...",
            label="Negative Prompt",
            value="low quality, worst quality, blurry, jpeg artifacts, watermark, text, logo, signature, cropped, duplicate, bad anatomy, extra fingers, missing fingers, fused fingers, deformed hands, mutated limbs, crossed eyes, lazy eye, asymmetrical face, malformed ears, low resolution, noisy, oversaturated, underexposed, overexposed, flat lighting, plastic skin, low detail, bad perspective, floating objects without support, duplicate objects, incorrect reflections, stretched proportions,,, "
        )
    ],
    outputs="image",
    title="Simple SDXL Text-to-Image Generator",
    description="Enter a text prompt to generate an image using Stable Diffusion XL."
)

# NOTE ON THE ORIGINAL BUG:
# iface.launch(server_port=0, ...) crashed with ConnectError: [Errno 111] Connection refused.
# Root cause: with server_port=0 (meaning "let the OS pick any free port"), Gradio 6.19's
# launch() never resolved the real OS-assigned port into self.server_port/self.local_url
# before doing its own internal readiness check (an httpx GET to that local URL). So it
# ended up checking "http://127.0.0.1:0/", and port 0 is never a valid *connect* target --
# it's only meaningful for binding. That GET was refused every single time, deterministically
# (confirmed by direct testing), not a transient timing race.
#
# FIX: pick a free port ourselves (via a real socket bind/close) and pass that explicit
# port to server_port instead of 0. Then Gradio's self-check hits the real port and succeeds.

def find_free_port():
    s = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
    s.bind(("127.0.0.1", 0))
    port = s.getsockname()[1]
    s.close()
    return port

print("Launching Gradio interface locally...")

gradio_actual_port = None
max_attempts = 3
for attempt in range(1, max_attempts + 1):
    free_port = find_free_port()
    try:
        iface.launch(share=False, server_port=free_port, debug=False, prevent_thread_lock=True, pwa=False)
        gradio_actual_port = iface.server_port
        print(f"Gradio interface is running locally on port {gradio_actual_port}.")
        break
    except Exception as e:
        print(f"Attempt {attempt}/{max_attempts} failed to launch Gradio on port {free_port}: {e}")
        try:
            iface.close()
        except Exception:
            pass
        time.sleep(2)

if gradio_actual_port is None:
    raise RuntimeError("Failed to launch the Gradio server after multiple attempts.")

# Make gradio_actual_port globally accessible for the Cloudflare tunnel cell
globals()["gradio_actual_port"] = gradio_actual_port
print("Please run the next cell to start the Cloudflare tunnel and get the public URL.")


Setting up Gradio interface...
Launching Gradio interface locally...
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
Note: opening Chrome Inspector may crash demo inside Colab notebooks.
* To create a public link, set `share=True` in `launch()`.


<IPython.core.display.Javascript object>

Gradio interface is running locally on port 40879.
Please run the next cell to start the Cloudflare tunnel and get the public URL.


### Set up Hugging Face Token (Optional but Recommended)

To avoid warnings about unauthenticated requests and potentially get higher rate limits when downloading models from Hugging Face, it's recommended to set your Hugging Face token. You can get one from [Hugging Face settings](https://huggingface.co/settings/tokens).

Add your token as a user secret in Colab (click the '🔑' icon on the left panel) with the name `HF_TOKEN`. Then, run the following cell.

In [ ]:
# @title
import os
from google.colab import userdata

# Set your Hugging Face Token if it exists in Colab secrets
try:
    hf_token = userdata.get('HF_TOKEN')
    if hf_token is not None:
        os.environ['HF_TOKEN'] = hf_token
        print("Hugging Face Token loaded from Colab secrets.")
    else:
        print("Hugging Face Token not found in Colab secrets.")
except KeyError:
    print("Hugging Face Token not found in Colab secrets (KeyError). Model downloads might be slower or encounter rate limits.")
except Exception as e:
    print(f"An unexpected error occurred while trying to get HF_TOKEN: {e}")
    print("Model downloads might be slower or encounter rate limits.")

Hugging Face Token loaded from Colab secrets.


### Install Cloudflared and Start Tunnel

This cell will install `cloudflared`, which is a command-line tool for Cloudflare Tunnels, and then start a tunnel to expose your Gradio application to the internet via a public URL. This provides a more reliable public link than Gradio's `share=True`.

# **Link in this ⬇**

In [ ]:
# @title
# Install cloudflared
!curl -L --output cloudflared.deb https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared.deb

print("Cloudflared installed. Starting tunnel...")

# Start the cloudflared tunnel in the background
# It will expose the Gradio app running on PORT (7860 by default).
# The URL will be printed to stderr, so we capture it.

import subprocess
import re
import time
import os

# Access the globally set port from the Gradio launch cell
global gradio_actual_port
PORT = gradio_actual_port if 'gradio_actual_port' in globals() else 7860 # Fallback in case previous cell wasn't run

def start_cloudflared(port):
    try:
        print(f"Starting cloudflared tunnel for port {port}...")
        # Use --url to expose the local Gradio server
        tunnel_process = subprocess.Popen(['cloudflared', 'tunnel', '--url', f'http://127.0.0.1:{port}'], stdout=subprocess.PIPE, stderr=subprocess.PIPE, text=True)

        public_url = None
        start_time = time.time()

        # Wait for cloudflared to print the URL (it usually prints to stderr)
        while time.time() - start_time < 30: # Wait up to 30 seconds for the URL
            line = tunnel_process.stderr.readline()
            if "https://" in line and ".trycloudflare.com" in line:
                public_url = re.search(r"https://[^\s]+\.trycloudflare\.com", line).group(0)
                print(f"Cloudflare Tunnel URL: {public_url}")
                break
            if not line: # No output, might be stuck, or tunnel is up and silent
                break
            # Uncomment below to see all cloudflared output for debugging
            # print(f"[cloudflared] {line.strip()}")

        if not public_url:
            print("Could not find Cloudflare Tunnel URL. Check cloudflared output for errors.")
            print("Cloudflared stdout:")
            print(tunnel_process.stdout.read())
            print("Cloudflared stderr:")
            print(tunnel_process.stderr.read())

        # Ensure the tunnel process is managed if not explicitly terminated later
        # You might want to store tunnel_process in a global var if you need to stop it later

    except Exception as e:
        print(f"Error starting cloudflared tunnel: {e}")

# Call the function to start the tunnel with the correct Gradio port
start_cloudflared(PORT)

print(f"Please use the Cloudflare Tunnel URL above (for port {PORT}) to access the Gradio interface.")


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
  0     0    0     0    0     0      0      0 --:--:-- --:--:-- --:--:--     0
100 18.0M  100 18.0M    0     0  18.3M      0 --:--:-- --:--:-- --:--:-- 18.3M
Selecting previously unselected package cloudflared.
(Reading database ... 122403 files and directories currently installed.)
Preparing to unpack cloudflared.deb ...
Unpacking cloudflared (2026.7.1) ...
Setting up cloudflared (2026.7.1) ...
Processing triggers for man-db (2.10.2-1) ...
Cloudflared installed. Starting tunnel...
Starting cloudflared tunnel for port 40879...
Cloudflare Tunnel URL: https://clay-trips-colorado-dirt.trycloudflare.com
Please use the Cloudflare Tunnel URL above (for port 40879) to access the Gradio interface.
